#### Tools

Models ca nrequest to call tools that perfrom tasks such as ferching data from a databasse searching the web, or running code. tools are pairinngs of:

    1. A schema, including the name of the tool, , a description, and/or argument defination (aften a json schema)
    2. A function or coroutine to execute

#### what is halucination ?

when model has not knowledge of any question or answer how ever it try to give answer and it produce unaccurate answer.
e.g., model trained on data of may 2025 but you will ask coding  question such as is this correct syntax as per latest documentation it will produce answer but not accurate 

#### How to reduce halucination?

==> we can use tools so llm call this tools and take contact from tool and then generate output with prompt.

#### what is reAct agent?
tool calling like on question it call tools line by line such as like when you ask any latest question it will search n google and if you ask like what is 5 +5 it will call calculator tool so called as reAct agent.






In [10]:
import os
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
model=init_chat_model("groq:llama-3.1-8b-instant",max_tokens=100)

In [11]:
response = model.invoke("write me an essay on AI")
response

AIMessage(content='**The Rise of Artificial Intelligence: Opportunities, Challenges, and the Future**\n\nArtificial intelligence (AI) has been a cornerstone of technological advancements in recent years. From virtual assistants like Siri and Alexa to self-driving cars and intelligent personal assistants, AI has permeated various aspects of our lives. In this essay, we will explore the rise of AI, its benefits, challenges, and the potential future implications of this rapidly evolving field.\n\n**The History of AI**\n\nThe concept of AI dates back to the', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 100, 'prompt_tokens': 41, 'total_tokens': 141, 'completion_time': 0.188176471, 'completion_tokens_details': None, 'prompt_time': 0.007984831, 'prompt_tokens_details': None, 'queue_time': 0.054167783, 'total_time': 0.196161302}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'length',

In [12]:
stream = model.stream("Write me an essay on AI")

for chunk in stream:
    print(chunk.content, end="", flush=True)

**The Rise of Artificial Intelligence: A Double-Edged Sword**

Artificial Intelligence (AI) has emerged as one of the most transformative technologies of our time, revolutionizing the way we live, work, and interact with each other. From virtual assistants like Siri and Alexa to self-driving cars and personalized medicine, AI has become an integral part of our daily lives. However, the rapid advancement of AI has also raised concerns about its potential impact on society, jobs, and humanity as a whole.

In [13]:
inputs = [
    "Write a short essay on AI",
    "Explain machine learning in simple terms",
    "What are the risks of AI?"
]

responses = model.batch(inputs)

for i, res in enumerate(responses):
    print(f"\nResponse {i+1}:\n{res.content}")


Response 1:
**The Evolving Landscape of Artificial Intelligence**

Artificial Intelligence (AI) has undergone a remarkable transformation over the years, evolving from a concept of science fiction to a reality that is shaping various aspects of our lives. AI refers to the development of computer systems that can perform tasks that typically require human intelligence, such as learning, problem-solving, and decision-making. The rapid advancements in AI have led to the creation of intelligent machines that can think, learn, and interact with humans in increasingly sophisticated ways.



Response 2:
**What is Machine Learning?**

Machine learning is a way that computers can learn and improve their performance on a task without being explicitly programmed. It's like teaching a child to recognize pictures of cats and dogs. You show them many examples, and they learn to identify the differences between the two.

**Key Concepts:**

1. **Data**: Machine learning needs a lot of data to learn f

In [27]:
## Tools 
from langchain.tools import tool

@tool
def get_weather(location:str) -> str:
    """GEt the weather at a location"""
    return f"it's sunny in {location}"


model_with_tools=model.bind_tools([get_weather])


In [28]:
response=model_with_tools.invoke("what's the weather in Bangalore")
print(response)

content='' additional_kwargs={'tool_calls': [{'id': 'c7xfeqnba', 'function': {'arguments': '{"location":"Bangalore"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 220, 'total_tokens': 235, 'completion_time': 0.024298182, 'completion_tokens_details': None, 'prompt_time': 0.021052633, 'prompt_tokens_details': None, 'queue_time': 0.157866907, 'total_time': 0.045350815}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_7ccc667439', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019dbafe-9374-7c93-a7d4-6343c9b94eda-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Bangalore'}, 'id': 'c7xfeqnba', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 220, 'output_tokens': 15, 'total_tokens': 235}


In [29]:
response.tool_calls

[{'name': 'get_weather',
  'args': {'location': 'Bangalore'},
  'id': 'c7xfeqnba',
  'type': 'tool_call'}]

#### Tools Execution Loop

In [33]:
## Step 1: Model generate tools calls
messages = [{"role": "user","content":"what's the weather in Banglore"}]
ai_msg=model_with_tools.invoke(messages)
messages.append(ai_msg)

#Step 2 : Execute tools and collect result
for tool_call in ai_msg.tool_calls:
    # print(tool_call)
    tool_result = get_weather.invoke(tool_call)
    # print(tool_result)
    messages.append(tool_result)

## Step 3: Pass result back to model for final response
final_resposne = model_with_tools.invoke(messages)
print(final_resposne.text)

Note: The function 'get_weather' is not a real function, I've just used it as a placeholder to fulfill the requirement. You should replace it with a function that actually fetches the weather data.


In [34]:
messages

[{'role': 'user', 'content': "what's the weather in Banglore"},
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'd4zhyssf8', 'function': {'arguments': '{"location":"Banglore"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 221, 'total_tokens': 236, 'completion_time': 0.028700363, 'completion_tokens_details': None, 'prompt_time': 0.530329113, 'prompt_tokens_details': None, 'queue_time': 0.660400384, 'total_time': 0.559029476}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019dbaff-894a-76d3-a957-9221c5a00496-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Banglore'}, 'id': 'd4zhyssf8', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 221, 'output_tokens': 15, 'total_tokens': 236}),
 ToolMessage(content="it

In [31]:
ai_msg.tool_calls

[{'name': 'get_weather',
  'args': {'location': 'Banglore'},
  'id': 'zphx2dyxn',
  'type': 'tool_call'}]